# NF-UNSW-NB15-v3 Data Preprocessing, Resampling & Partitioning Pipeline (v2)

This notebook implements the updated modular workflow according to `dataset.txt`:
- **Step 1**: Pruning non-generalizable shortcut learning features (IPs, Ports, Timestamps) and deduplicating across all remaining features using DuckDB.
- **Step 2**: Stratified sampling (80/20) to extract a pristine 20% Holdout test set (`holdout_df`).
- **Step 3**: Stratified splitting of the 80% non-holdout dev set into Train (70%), Validation (10%), and Test (20%). All downstream transformations are strictly performed on the train set only.
- **Steps 4 to 6**: Invoking the `NFUNSWAttackResampler` class from `support_scripts/new_nf_unsw_pipeline.py`:
  - **Step 4**: Disaggregating benign and attack flows.
  - **Step 5**: Calculating Synthetic Deficit ($t_c = \max(n_c, \min(t_{\max}, y_{\max} \cdot n_c))$, $n_{\text{syn}} = \max(0, t_c - n_c)$) with default $t_{\max} = 10\text{k}, y_{\max} = 25$.
  - **Step 6**: Normalizing continuous features, formulating the dict of $t_c$ target counts, and oversampling attacks via SMOTE-NC (no further heuristic post-processing).

In [ ]:
import os
import sys
import time
from pathlib import Path

# Mitigate OpenMP conflicts across libraries
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["OMP_NUM_THREADS"] = "4"
os.environ["MKL_NUM_THREADS"] = "4"

import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

# Locate project base directory and datasets
curr_dir = Path.cwd()
repo_roots = [
    curr_dir.parents[2] if len(curr_dir.parents) >= 3 else curr_dir,
    curr_dir.parents[1] if len(curr_dir.parents) >= 2 else curr_dir,
    curr_dir.parent,
    curr_dir,
    Path("/run/media/thanh/Data/Project A/duc_thanh"),
    Path("/run/media/thanh/Ventoy/Project A/duc_thanh"),
]

base_dir = None
for r in repo_roots:
    if (r / "datasets/NetFlow").exists():
        base_dir = r.resolve()
        break

if base_dir is None:
    raise FileNotFoundError("Could not resolve workspace root containing 'datasets/NetFlow'!")

raw_csv_candidates = list(base_dir.glob("**/NF-UNSW-NB15-v3.csv"))
if not raw_csv_candidates:
    raise FileNotFoundError(f"Raw NF-UNSW-NB15-v3.csv not found under {base_dir}!")
raw_csv_path = raw_csv_candidates[0]

final_datasets_dir = base_dir / "datasets/final datasets"
holdout_dir = final_datasets_dir / "holdout_set"
dev_splits_dir = final_datasets_dir / "dev_splits"
ids_multi_dir = final_datasets_dir / "IDS-multiclass-classification"

for d in [final_datasets_dir, holdout_dir, dev_splits_dir, ids_multi_dir]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Workspace Base:         {base_dir}")
print(f"Raw Input CSV:          {raw_csv_path}")
print(f"Final Datasets Dir:     {final_datasets_dir}")

## Step 1: Pruning Non-Generalizable Shortcut Learning Features & Deduplication

To eliminate shortcut learning (the Clever Hans effect), we prune non-generalizable network identifiers:
- `IPV4_SRC_ADDR`, `L4_SRC_PORT`
- `IPV4_DST_ADDR`, `L4_DST_PORT`
- `FLOW_START_MILLISECONDS`, `FLOW_END_MILLISECONDS`

We then deduplicate across all remaining 47 features and labels via DuckDB.

In [ ]:
# ----------------------------------------------------------------------------
# Step 1: Feature Pruning & Deduplication via DuckDB
# ----------------------------------------------------------------------------
t0 = time.time()
SHORTCUT_COLS = [
    "IPV4_SRC_ADDR",
    "L4_SRC_PORT",
    "IPV4_DST_ADDR",
    "L4_DST_PORT",
    "FLOW_START_MILLISECONDS",
    "FLOW_END_MILLISECONDS",
]

exclude_sql = f"EXCLUDE ({', '.join(SHORTCUT_COLS)})"
raw_path_str = str(raw_csv_path).replace("\\", "/")

con = duckdb.connect()
con.execute("PRAGMA threads=6;")
con.execute("PRAGMA memory_limit='10GB';")
con.execute("PRAGMA preserve_insertion_order=false;")

# Total raw record count
raw_total_count = con.execute(f"SELECT COUNT(*) FROM read_csv_auto('{raw_path_str}');").fetchone()[0]

# Deduplicate across all retained features
con.execute(f"""
    CREATE OR REPLACE TABLE unsw_clean AS
    SELECT DISTINCT * {exclude_sql}
    FROM read_csv_auto('{raw_path_str}');
""")

dedup_total_count = con.execute("SELECT COUNT(*) FROM unsw_clean;").fetchone()[0]
benign_dedup_count = con.execute("SELECT COUNT(*) FROM unsw_clean WHERE Label = 0;").fetchone()[0]
attack_dedup_count = con.execute("SELECT COUNT(*) FROM unsw_clean WHERE Label = 1;").fetchone()[0]
removed_duplicates = raw_total_count - dedup_total_count

print("=== Step 1: Shortcut Pruning & Deduplication Summary ===")
print(f"  Raw Flows:             {raw_total_count:,}")
print(f"  Deduplicated Flows:    {dedup_total_count:,} (Removed {removed_duplicates:,} duplicates, "
      f"{(removed_duplicates / raw_total_count) * 100:.2f}%)")
print(f"  Benign Flows:          {benign_dedup_count:,} ({benign_dedup_count / dedup_total_count * 100:.2f}%)")
print(f"  Attack Flows:          {attack_dedup_count:,} ({attack_dedup_count / dedup_total_count * 100:.2f}%)")
print(f"  Pruned Features ({len(SHORTCUT_COLS)}):    {SHORTCUT_COLS}")
print(f"  Step 1 Completed in {time.time() - t0:.2f}s")

# Load clean deduplicated dataset into Pandas
df_clean = con.execute("SELECT * FROM unsw_clean;").df()
feature_cols = [c for c in df_clean.columns if c not in ["Label", "Attack"]]
print(f"  Remaining Features:    {len(feature_cols)} features")

## Step 2: Stratified Holdout Partitioning (80/20)

We extract a pristine 20% holdout test set using stratified sampling on `Attack`.
This guarantees all attack categories are proportionally preserved in the holdout distribution.
All downstream development, training, and oversampling strictly use the remaining 80% development set (`dev_df`).

In [ ]:
# ----------------------------------------------------------------------------
# Step 2: Stratified Sampling for Pristine Holdout Test Set (20%)
# ----------------------------------------------------------------------------
t0 = time.time()
RANDOM_STATE = 42

dev_df, holdout_df = train_test_split(
    df_clean,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=df_clean["Attack"]
)
dev_df = dev_df.reset_index(drop=True)
holdout_df = holdout_df.reset_index(drop=True)

print("=== Step 2: Holdout Partitioning Summary ===")
print(f"  Non-Holdout Dev Set (80%): {len(dev_df):,} flows")
print(f"  Holdout Test Set    (20%): {len(holdout_df):,} flows")

# Materialize pristine holdout test set
holdout_path = holdout_dir / "NF-UNSW-NB15-v3.csv"
holdout_path_str = str(holdout_path).replace("\\", "/")
con.execute(f"COPY holdout_df TO '{holdout_path_str}' (HEADER, DELIMITER ',');")
print(f"  Materialized Holdout:      {holdout_path.name} ({holdout_path.stat().st_size / (1024**2):.2f} MB)")
print(f"  Step 2 Completed in {time.time() - t0:.2f}s")

## Step 3: Stratified Train / Val / Test Split of the 80% Dev Set (70 / 10 / 20)

We partition the 80% non-holdout development set into:
- **Train (70% of dev)**: Target partition for training and subsequent oversampling transformations.
- **Validation (10% of dev)**: For threshold tuning and hyperparameter selection.
- **Test (20% of dev)**: For unbiased evaluation on the dev distribution.

All downstream transformations are strictly confined to the training set (`train_df`).

In [ ]:
# ----------------------------------------------------------------------------
# Step 3: Split 80% Dev Set into Train (70%), Val (10%), Test (20%)
# ----------------------------------------------------------------------------
t0 = time.time()

# 1. Extract 20% Test partition from dev_df
train_val_df, dev_test_df = train_test_split(
    dev_df,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=dev_df["Attack"]
)

# 2. Extract 10% of dev_df as Val from the remaining 80% (10/80 = 0.125)
train_df, dev_val_df = train_test_split(
    train_val_df,
    test_size=0.125,
    random_state=RANDOM_STATE,
    stratify=train_val_df["Attack"]
)

train_df = train_df.reset_index(drop=True)
dev_val_df = dev_val_df.reset_index(drop=True)
dev_test_df = dev_test_df.reset_index(drop=True)

total_dev = len(dev_df)
print("=== Step 3: Dev Set Partitions ===")
print(f"  Train Partition (70% of Dev): {len(train_df):,} flows ({len(train_df) / total_dev * 100:.2f}%)")
print(f"  Val Partition   (10% of Dev): {len(dev_val_df):,} flows ({len(dev_val_df) / total_dev * 100:.2f}%)")
print(f"  Test Partition  (20% of Dev): {len(dev_test_df):,} flows ({len(dev_test_df) / total_dev * 100:.2f}%)")

# Save partitions
for name, split_df in [("train", train_df), ("val", dev_val_df), ("test", dev_test_df)]:
    out_p = dev_splits_dir / f"NF-UNSW-NB15-v3_{name}.csv"
    out_p_str = str(out_p).replace("\\", "/")
    con.execute(f"COPY split_df TO '{out_p_str}' (HEADER, DELIMITER ',');")
    print(f"  Saved {name.capitalize()} Partition: {out_p.name} ({out_p.stat().st_size / (1024**2):.2f} MB)")

print(f"  Step 3 Completed in {time.time() - t0:.2f}s")

## Steps 4 to 6: Invoking `NFUNSWAttackResampler` from Support Script

We now import and invoke the `NFUNSWAttackResampler` class from `support_scripts/new_nf_unsw_pipeline.py`.
The class executes:
- **Step 4**: Disaggregates `train_benign` and `train_attacks`.
- **Step 5**: Calculates the Synthetic Deficit:
  $$t_c = \max\left(n_c, \min\left(t_{\max}, y_{\max} \cdot n_c\right)\right)$$
  $$n_{\text{syn}} = \max\left(0, t_c - n_c\right)$$
  Constructs a dictionary of target counts $t_c$ (`dict_tc`) for each class.
- **Step 6**: Normalizes continuous features via `MinMaxScaler`, passes `dict_tc` directly to `SMOTENC` with categorical indices, and exports resampled datasets without ad-hoc post-filtering.

In [ ]:
# ----------------------------------------------------------------------------
# Steps 4 to 6: Calling NFUNSWAttackResampler Class
# ----------------------------------------------------------------------------
# Add support_scripts directory to Python module search path
support_candidates = [
    curr_dir,
    curr_dir.parent,
    curr_dir.parent / "components",
    base_dir / "SAP-DCG" if base_dir else curr_dir,
    base_dir / "SAP-DCG/components" if base_dir else curr_dir,
    curr_dir / "../support_scripts",
    curr_dir.parent / "support_scripts",
    base_dir / "notebooks/data_processing/support_scripts",
]
for sc in support_candidates:
    if sc.exists() and str(sc.resolve()) not in sys.path:
        sys.path.append(str(sc.resolve()))

try:
    from intra_attack_resampler import IntraAttackResampler
except ImportError:
    from new_nf_unsw_pipeline import NFUNSWAttackResampler

# 1. Instantiate the resampler class
resampler = IntraAttackResampler(
    t_max=10000,
    y_max=25,
    random_state=42,
    scaler_type="minmax",
    final_dir=final_datasets_dir
)

# 2. Execute Steps 4-6 on train_df
resampled_results = resampler.fit_resample(train_df=train_df, export=True)

# Access returned objects
train_benign = resampled_results["train_benign"]
train_attacks_raw = resampled_results["train_attacks_raw"]
df_attacks_smotenc = resampled_results["df_attacks_smotenc"]
df_attacks_smotenc_norm = resampled_results["df_attacks_smotenc_norm"]
df_train_resampled = resampled_results["df_train_resampled"]
dict_tc = resampled_results["dict_tc"]
syn_deficit = resampled_results["syn_deficit"]
df_deficit_summary = resampled_results["deficit_summary"]

print("\n=== Synthetic Deficit Calculation Breakdown (Step 5) ===")
display(df_deficit_summary)

## Diagnostic Evaluation & Verification

We verify the class balance before and after SMOTE-NC oversampling, and inspect the catalog of exported datasets in `datasets/final datasets/`.

In [ ]:
# ----------------------------------------------------------------------------
# Comparison: Attack Class Distribution Before vs. After SMOTE-NC
# ----------------------------------------------------------------------------
df_comp = pd.DataFrame({
    "Original Train Count": train_attacks_raw["Attack"].value_counts(),
    "Target t_c": pd.Series(dict_tc),
    "Resampled Count": df_attacks_smotenc["Attack"].value_counts()
}).fillna(0).astype(int)

df_comp["Synthetic Generated"] = df_comp["Resampled Count"] - df_comp["Original Train Count"]
df_comp["Expansion Rate"] = (df_comp["Resampled Count"] / df_comp["Original Train Count"]).round(2).astype(str) + "x"

print("=== Attack Class Distribution: Before vs. After Resampling ===")
display(df_comp)

# Visualize Distribution Comparison
fig, ax = plt.subplots(figsize=(12, 5))
df_plot = df_comp[["Original Train Count", "Resampled Count"]].reset_index().melt(
    id_vars="index", var_name="Dataset", value_name="Count"
)
df_plot.rename(columns={"index": "Attack Category"}, inplace=True)

sns.barplot(data=df_plot, x="Attack Category", y="Count", hue="Dataset", palette=["#3498db", "#e74c3c"], ax=ax)
ax.set_title("NF-UNSW-NB15-v3 Attack Classes: Before vs. After SMOTE-NC Resampling", fontweight="bold", fontsize=12)
ax.set_yscale("log")
ax.set_ylabel("Flow Count (Log Scale)", fontweight="bold")
ax.set_xlabel("Attack Category", fontweight="bold")
plt.xticks(rotation=30, ha="right")
plt.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
# ----------------------------------------------------------------------------
# Final Catalog of Exported Datasets
# ----------------------------------------------------------------------------
catalog_records = []
for csv_file in sorted(final_datasets_dir.glob("**/*.csv")):
    size_mb = csv_file.stat().st_size / (1024 ** 2)
    csv_file_str = str(csv_file).replace("\\", "/")
    row_count = con.execute(f"SELECT COUNT(*) FROM read_csv_auto('{csv_file_str}');").fetchone()[0]
    catalog_records.append({
        "Relative Path": str(csv_file.relative_to(final_datasets_dir)),
        "Flow Rows": f"{row_count:,}",
        "Size (MB)": f"{size_mb:.2f}"
    })

df_catalog = pd.DataFrame(catalog_records)
print("=== Catalog of Exported Final Datasets ===")
display(df_catalog)